In [ ]:
!pip install sdmetrics

In [1]:
from sdmetrics.reports.single_table import QualityReport
import pandas as pd


import json

def load_config(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)
    
def load_table(data_path):
    df = pd.read_csv(data_path)
    return df

In [3]:

def build_metadata_from_config(df, config, dataset='adult'):
    metadata = {
        "columns": {}
    }
    num_cols = set(config["num_col_idx"])
    cat_cols = set(config["cat_col_idx"])
    target_cols = set(config["target_col_idx"])

    for i, col in enumerate(df.columns):
        if i in num_cols:
            sdtype = "numerical"
        elif i in cat_cols:
            sdtype = "categorical"
        elif i in target_cols and dataset != 'beijing':
            sdtype = "categorical"
        elif i in target_cols and dataset == 'beijing':
            sdtype = "numerical"

        metadata["columns"][col] = {"sdtype": sdtype}

    return metadata
    

In [4]:
def calculate_quality_scores(real_df, pred_df, config, dataset='adult'):
    metadata = build_metadata_from_config(real_df, config, dataset)
    report = QualityReport()
    report.generate(real_df, pred_df, metadata)
    properties = report.get_properties().copy()

    column_shapes_score = properties.loc[
        properties["Property"] == "Column Shapes", "Score"
    ].iloc[0]

    column_pair_trends_score = properties.loc[
        properties["Property"] == "Column Pair Trends", "Score"
    ].iloc[0]

    overall_score = report.get_score()

    per_column_scores = report.get_details("Column Shapes").copy()
    per_column_scores = per_column_scores[["Column", "Score"]].rename(
        columns={"Score": "shape_score"}
    )

    pair_details = report.get_details("Column Pair Trends").copy()

    result = {
        "dataset": dataset,
        "column_shapes_score": column_shapes_score,
        "column_pair_trends_score": column_pair_trends_score,
        "overall_score": overall_score,
        "per_column_scores": per_column_scores,
        "pair_details": pair_details,
        "properties": properties,
        "report": report,
    }

    return result


In [ ]:
!ls

LICENSE       data          tabsyn        test.py
README.md     metrics.ipynb tadiff


In [31]:
!ls

LICENSE       data          tabcsdi       tadiff
README.md     metrics.ipynb tabsyn        test.py


# Adult

In [7]:
def calc(dataset):
    model_paths = {
        "tabcsdi": f"./tabcsdi/prediction_{dataset}.csv",
        "tabsyn": f"./tabsyn/prediction_{dataset}.csv",
        "tabdiff": f"./tabdiff/prediction_{dataset}.csv",
        "catboost": f"./catboost/prediction_{dataset}.csv"
    }
    print(len(model_paths))

    config = load_config(f"./data/{dataset}/info.json")
    real_df = load_table(f"./data/{dataset}/test.csv")

    all_results = {}
    summary_rows = []
    per_column_rows = []
    pair_rows = []

    for model_name, pred_path in model_paths.items():
        pred_df = load_table(pred_path)

        result = calculate_quality_scores(
            real_df=real_df,
            pred_df=pred_df,
            config=config,
            dataset=dataset
        )

        all_results[model_name] = result


        summary_rows.append({
            "dataset": dataset,
            "model": model_name,
            "Column Shapes Score": result["column_shapes_score"],
            "Column Pair Trends Score": result["column_pair_trends_score"],
            "Overall Score": result["overall_score"],
        })

        tmp_col = result["per_column_scores"].copy()
        tmp_col["dataset"] =dataset
        tmp_col["model"] = model_name
        per_column_rows.append(tmp_col)

        tmp_pair = result["pair_details"].copy()
        tmp_pair["dataset"] = dataset
        tmp_pair["model"] = model_name
        pair_rows.append(tmp_pair)
    return summary_rows, per_column_rows, pair_rows
    

In [8]:
summary_rows, per_column_rows, pair_rows = calc('adult')
summary_df = pd.DataFrame(summary_rows)
per_column_df = pd.concat(per_column_rows, ignore_index=True)
pair_df = pd.concat(pair_rows, ignore_index=True)

4
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 229.27it/s]|
Column Shapes Score: 84.16%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 464.17it/s]|
Column Pair Trends Score: 87.13%

Overall Score (Average): 85.65%

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 230.86it/s]|
Column Shapes Score: 96.97%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 476.86it/s]|
Column Pair Trends Score: 90.17%

Overall Score (Average): 93.57%

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 279.71it/s]|
Column Shapes Score: 98.9%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 472.77it/s]|
Column Pair Trends Score: 85.83%

Overall Score (Average): 92.36%

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 191.70it/s]|
Column Shapes Score: 73.54%

(2/2) Evaluating Column

In [9]:
summary_df

,dataset,model,Column Shapes Score,Column Pair Trends Score,Overall Score
0,adult,tabcsdi,0.841599,0.871332,0.856465
1,adult,tabsyn,0.969654,0.901698,0.935676
2,adult,tabdiff,0.988981,0.858311,0.923646
3,adult,catboost,0.735360,0.750426,0.742893


In [10]:
per_column_df

,Column,shape_score,dataset,model
0,age,0.944414,adult,tabcsdi
1,workclass,0.993735,adult,tabcsdi
2,fnlwgt,0.947485,adult,tabcsdi
3,education,0.983416,adult,tabcsdi
4,education.num,0.752779,adult,tabcsdi
5,marital.status,0.981021,adult,tabcsdi
6,occupation,0.942878,adult,tabcsdi
7,relationship,0.972852,adult,tabcsdi
8,race,0.992629,adult,tabcsdi
9,sex,0.998280,adult,tabcsdi


In [11]:
pair_df.dropna(subset=["Score"])

,Column 1,Column 2,Metric,Score,Real Correlation,Synthetic Correlation,Real Association,Meets Threshold?,dataset,model
13,age,income,ContingencySimilarity,0.938763,NaN,NaN,0.315142,True,adult,tabcsdi
18,workclass,occupation,ContingencySimilarity,0.861925,NaN,NaN,0.400611,True,adult,tabcsdi
39,education,education.num,ContingencySimilarity,0.564462,NaN,NaN,1.000000,True,adult,tabcsdi
49,education,income,ContingencySimilarity,0.968307,NaN,NaN,0.360356,True,adult,tabcsdi
59,education.num,income,ContingencySimilarity,0.587925,NaN,NaN,0.354417,True,adult,tabcsdi
61,marital.status,relationship,ContingencySimilarity,0.894233,NaN,NaN,0.488974,True,adult,tabcsdi
63,marital.status,sex,ContingencySimilarity,0.962902,NaN,NaN,0.455158,True,adult,tabcsdi
68,marital.status,income,ContingencySimilarity,0.960997,NaN,NaN,0.450240,True,adult,tabcsdi
71,occupation,sex,ContingencySimilarity,0.871445,NaN,NaN,0.424559,True,adult,tabcsdi
76,occupation,income,ContingencySimilarity,0.910939,NaN,NaN,0.347587,True,adult,tabcsdi


# Magic

In [12]:
summary_rows, per_column_rows, pair_rows = calc('magic')
summary_df = pd.DataFrame(summary_rows)
per_column_df = pd.concat(per_column_rows, ignore_index=True)
pair_df = pd.concat(pair_rows, ignore_index=True)

4
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 540.09it/s]|
Column Shapes Score: 96.11%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 730.77it/s]|
Column Pair Trends Score: 95.12%

Overall Score (Average): 95.62%

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 1057.52it/s]|
Column Shapes Score: 97.16%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 907.88it/s]|
Column Pair Trends Score: 94.91%

Overall Score (Average): 96.04%

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 1087.91it/s]|
Column Shapes Score: 95.86%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 669.79it/s]|
Column Pair Trends Score: 92.46%

Overall Score (Average): 94.16%

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 1030.33it/s]|
Column Shapes Score: 84.35%

(2/2) Evaluating Column P

In [13]:
summary_df

,dataset,model,Column Shapes Score,Column Pair Trends Score,Overall Score
0,magic,tabcsdi,0.961141,0.951245,0.956193
1,magic,tabsyn,0.971561,0.949142,0.960352
2,magic,tabdiff,0.958608,0.924589,0.941598
3,magic,catboost,0.843466,0.920012,0.881739


In [14]:
per_column_df

,Column,shape_score,dataset,model
0,Length,0.973712,magic,tabcsdi
1,Width,0.958465,magic,tabcsdi
2,Size,0.981073,magic,tabcsdi
3,Conc,0.984227,magic,tabcsdi
4,Conc1,0.986330,magic,tabcsdi
5,Asym,0.923239,magic,tabcsdi
6,M3Long,0.943218,magic,tabcsdi
7,M3Trans,0.911672,magic,tabcsdi
8,Alpha,0.964774,magic,tabcsdi
9,Dist,0.967403,magic,tabcsdi


In [15]:
pair_df.dropna(subset=["Score"])

,Column 1,Column 2,Metric,Score,Real Correlation,Synthetic Correlation,Real Association,Meets Threshold?,dataset,model
0,Length,Width,CorrelationSimilarity,0.981540,0.752390,0.789309,NaN,True,magic,tabcsdi
1,Length,Size,CorrelationSimilarity,0.996430,0.698769,0.705908,NaN,True,magic,tabcsdi
2,Length,Conc,CorrelationSimilarity,0.994738,-0.625760,-0.636283,NaN,True,magic,tabcsdi
3,Length,Conc1,CorrelationSimilarity,0.984604,-0.588086,-0.618877,NaN,True,magic,tabcsdi
9,Length,class,ContingencySimilarity,0.941640,NaN,NaN,0.358254,True,magic,tabcsdi
10,Width,Size,CorrelationSimilarity,0.995318,0.684998,0.694362,NaN,True,magic,tabcsdi
11,Width,Conc,CorrelationSimilarity,0.994299,-0.584398,-0.595799,NaN,True,magic,tabcsdi
12,Width,Conc1,CorrelationSimilarity,0.984764,-0.555381,-0.585853,NaN,True,magic,tabcsdi
18,Width,class,ContingencySimilarity,0.869085,NaN,NaN,0.335960,True,magic,tabcsdi
19,Size,Conc,CorrelationSimilarity,0.999536,-0.854080,-0.853152,NaN,True,magic,tabcsdi


# Shoppers

In [16]:
summary_rows, per_column_rows, pair_rows = calc('shoppers')
summary_df = pd.DataFrame(summary_rows)
per_column_df = pd.concat(per_column_rows, ignore_index=True)
pair_df = pd.concat(pair_rows, ignore_index=True)

4
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 18/18 [00:00<00:00, 1349.11it/s]|
Column Shapes Score: 76.23%

(2/2) Evaluating Column Pair Trends: |██████████| 153/153 [00:00<00:00, 974.43it/s]|
Column Pair Trends Score: 89.22%

Overall Score (Average): 82.73%

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 18/18 [00:00<00:00, 2159.85it/s]|
Column Shapes Score: 95.67%

(2/2) Evaluating Column Pair Trends: |██████████| 153/153 [00:00<00:00, 1334.34it/s]|
Column Pair Trends Score: 90.19%

Overall Score (Average): 92.93%

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 18/18 [00:00<00:00, 2215.17it/s]|
Column Shapes Score: 95.46%

(2/2) Evaluating Column Pair Trends: |██████████| 153/153 [00:00<00:00, 1342.69it/s]|
Column Pair Trends Score: 85.75%

Overall Score (Average): 90.61%

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 18/18 [00:00<00:00, 2025.42it/s]|
Column Shapes Score: 65.12%

(2/2) Evaluating

In [17]:
summary_df

,dataset,model,Column Shapes Score,Column Pair Trends Score,Overall Score
0,shoppers,tabcsdi,0.762323,0.892222,0.827273
1,shoppers,tabsyn,0.956655,0.901875,0.929265
2,shoppers,tabdiff,0.954627,0.857503,0.906065
3,shoppers,catboost,0.651167,0.818015,0.734591


In [18]:
per_column_df

,Column,shape_score,dataset,model
0,Administrative,0.579075,shoppers,tabcsdi
1,Administrative_Duration,0.682887,shoppers,tabcsdi
2,Informational,0.328467,shoppers,tabcsdi
3,Informational_Duration,0.243309,shoppers,tabcsdi
4,ProductRelated,0.909165,shoppers,tabcsdi
...,...,...,...,...
67,Region,0.397405,shoppers,catboost
68,TrafficType,0.733982,shoppers,catboost
69,VisitorType,0.927818,shoppers,catboost
70,Weekend,0.768856,shoppers,catboost


In [19]:
pair_df.dropna(subset=["Score"])

,Column 1,Column 2,Metric,Score,Real Correlation,Synthetic Correlation,Real Association,Meets Threshold?,dataset,model
0,Administrative,Administrative_Duration,CorrelationSimilarity,0.960437,0.593842,0.514716,NaN,True,shoppers,tabcsdi
33,Informational,Informational_Duration,CorrelationSimilarity,0.882085,0.687845,0.452014,NaN,True,shoppers,tabcsdi
62,ProductRelated,ProductRelated_Duration,CorrelationSimilarity,0.888710,0.883052,0.660472,NaN,True,shoppers,tabcsdi
87,BounceRates,ExitRates,CorrelationSimilarity,0.989136,0.918279,0.896552,NaN,True,shoppers,tabcsdi
116,PageValues,Revenue,ContingencySimilarity,0.885645,NaN,NaN,0.552291,True,shoppers,tabcsdi
132,OperatingSystems,Browser,ContingencySimilarity,0.783455,NaN,NaN,0.616624,True,shoppers,tabcsdi
134,OperatingSystems,TrafficType,ContingencySimilarity,0.846715,NaN,NaN,0.354013,True,shoppers,tabcsdi
135,OperatingSystems,VisitorType,ContingencySimilarity,0.916464,NaN,NaN,0.386113,True,shoppers,tabcsdi
139,Browser,TrafficType,ContingencySimilarity,0.840227,NaN,NaN,0.326741,True,shoppers,tabcsdi
140,Browser,VisitorType,ContingencySimilarity,0.940795,NaN,NaN,0.498952,True,shoppers,tabcsdi


# Default

In [20]:
summary_rows, per_column_rows, pair_rows = calc('default')
summary_df = pd.DataFrame(summary_rows)
per_column_df = pd.concat(per_column_rows, ignore_index=True)
pair_df = pd.concat(pair_rows, ignore_index=True)

4
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 24/24 [00:00<00:00, 721.35it/s]|
Column Shapes Score: 90.52%

(2/2) Evaluating Column Pair Trends: |██████████| 276/276 [00:00<00:00, 1179.30it/s]|
Column Pair Trends Score: 93.59%

Overall Score (Average): 92.05%

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 24/24 [00:00<00:00, 1026.61it/s]|
Column Shapes Score: 98.09%

(2/2) Evaluating Column Pair Trends: |██████████| 276/276 [00:00<00:00, 1227.24it/s]|
Column Pair Trends Score: 97.64%

Overall Score (Average): 97.87%

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 24/24 [00:00<00:00, 949.45it/s]|
Column Shapes Score: 97.57%

(2/2) Evaluating Column Pair Trends: |██████████| 276/276 [00:00<00:00, 1100.50it/s]|
Column Pair Trends Score: 91.7%

Overall Score (Average): 94.63%

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 24/24 [00:00<00:00, 540.70it/s]|
Column Shapes Score: 83.76%

(2/2) Evaluating Co

In [21]:
summary_df

,dataset,model,Column Shapes Score,Column Pair Trends Score,Overall Score
0,default,tabcsdi,0.905153,0.935870,0.920512
1,default,tabsyn,0.980903,0.976423,0.978663
2,default,tabdiff,0.975708,0.916990,0.946349
3,default,catboost,0.837597,0.953961,0.895779


In [22]:
pair_df

,Column 1,Column 2,Metric,Score,Real Correlation,Synthetic Correlation,Real Association,Meets Threshold?,dataset,model
0,LIMIT_BAL,SEX,ContingencySimilarity,NaN,NaN,NaN,0.099949,False,default,tabcsdi
1,LIMIT_BAL,EDUCATION,ContingencySimilarity,NaN,NaN,NaN,0.135725,False,default,tabcsdi
2,LIMIT_BAL,MARRIAGE,ContingencySimilarity,NaN,NaN,NaN,0.080955,False,default,tabcsdi
3,LIMIT_BAL,AGE,CorrelationSimilarity,NaN,0.137718,NaN,NaN,False,default,tabcsdi
4,LIMIT_BAL,PAY_0,ContingencySimilarity,NaN,NaN,NaN,0.111955,False,default,tabcsdi
...,...,...,...,...,...,...,...,...,...,...
1099,PAY_AMT4,PAY_AMT6,CorrelationSimilarity,NaN,0.284112,NaN,NaN,False,default,catboost
1100,PAY_AMT4,default payment next month,ContingencySimilarity,NaN,NaN,NaN,0.049440,False,default,catboost
1101,PAY_AMT5,PAY_AMT6,CorrelationSimilarity,NaN,0.231412,NaN,NaN,False,default,catboost
1102,PAY_AMT5,default payment next month,ContingencySimilarity,NaN,NaN,NaN,0.047362,False,default,catboost


# Beijing

In [23]:
summary_rows, per_column_rows, pair_rows = calc('beijing')
summary_df = pd.DataFrame(summary_rows)
per_column_df = pd.concat(per_column_rows, ignore_index=True)
pair_df = pd.concat(pair_rows, ignore_index=True)

4
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 478.83it/s]|
Column Shapes Score: 56.98%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 937.15it/s]|
Column Pair Trends Score: 52.22%

Overall Score (Average): 54.6%

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 563.86it/s]|
Column Shapes Score: 95.85%



/opt/anaconda3/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:531: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)


(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 927.27it/s]|
Column Pair Trends Score: 90.82%

Overall Score (Average): 93.34%

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 772.36it/s]|
Column Shapes Score: 96.21%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 1007.36it/s]|
Column Pair Trends Score: 86.2%

Overall Score (Average): 91.21%

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 12/12 [00:00<00:00, 684.78it/s]|
Column Shapes Score: 77.56%

(2/2) Evaluating Column Pair Trends: |██████████| 66/66 [00:00<00:00, 1014.14it/s]|
Column Pair Trends Score: 82.92%

Overall Score (Average): 80.24%



In [24]:
summary_df

,dataset,model,Column Shapes Score,Column Pair Trends Score,Overall Score
0,beijing,tabcsdi,0.569824,0.522207,0.546015
1,beijing,tabsyn,0.958493,0.908235,0.933364
2,beijing,tabdiff,0.962145,0.862049,0.912097
3,beijing,catboost,0.775583,0.829211,0.802397


In [25]:
per_column_df

,Column,shape_score,dataset,model
0,year,0.973659,beijing,tabcsdi
1,month,0.894157,beijing,tabcsdi
2,day,0.857280,beijing,tabcsdi
3,hour,0.899425,beijing,tabcsdi
4,pm2.5,0.576389,beijing,tabcsdi
5,DEWP,0.494013,beijing,tabcsdi
6,TEMP,0.459052,beijing,tabcsdi
7,PRES,0.275623,beijing,tabcsdi
8,cbwd,0.774665,beijing,tabcsdi
9,Iws,0.428640,beijing,tabcsdi


In [26]:
pair_df

,Column 1,Column 2,Metric,Score,Real Correlation,Synthetic Correlation,Real Association,Meets Threshold?,dataset,model
0,year,month,ContingencySimilarity,NaN,NaN,NaN,0.054337,False,beijing,tabcsdi
1,year,day,ContingencySimilarity,NaN,NaN,NaN,0.083443,False,beijing,tabcsdi
2,year,hour,ContingencySimilarity,NaN,NaN,NaN,0.072217,False,beijing,tabcsdi
3,year,pm2.5,ContingencySimilarity,NaN,NaN,NaN,0.064373,False,beijing,tabcsdi
4,year,DEWP,ContingencySimilarity,NaN,NaN,NaN,0.085296,False,beijing,tabcsdi
...,...,...,...,...,...,...,...,...,...,...
259,cbwd,Is,ContingencySimilarity,NaN,NaN,NaN,0.032689,False,beijing,catboost
260,cbwd,Ir,ContingencySimilarity,NaN,NaN,NaN,0.052604,False,beijing,catboost
261,Iws,Is,CorrelationSimilarity,NaN,0.014429,NaN,NaN,False,beijing,catboost
262,Iws,Ir,CorrelationSimilarity,NaN,-0.028449,NaN,NaN,False,beijing,catboost
